# Phase 1 — Reproducible Four-Modality Engagement Experiment

This notebook audits and aligns EEG, Eye, Face/OpenFace, and HR prepared session tables; maps the original labels to Low/Medium/High; performs training-fold-only preprocessing; and compares **RBF-SVM**, **Random Forest**, and a **gated four-encoder MLP** using nested participant-grouped validation. It saves the exact out-of-fold predictions, metrics, tables, configuration, and plots for Phase 2.

**Important:** the prepared files are session-level summary features, not ordered time windows or face images. This notebook therefore uses tabular encoders; it does not claim LSTM/GRU temporal learning, CNN image extraction, or Differential Entropy from raw EEG. Validation results are cross-validation estimates, not an untouched final test score. No accuracy is promised in advance.

In [2]:
# 1. Imports, reproducibility, and runtime
import os, re, json, time, gc, random, warnings, zipfile, hashlib, platform
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from IPython.display import display
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
    balanced_accuracy_score, matthews_corrcoef, confusion_matrix, classification_report)
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
warnings.filterwarnings('ignore', category=FutureWarning)
MODALITIES = ['EEG', 'Eye', 'Face', 'HR']
CLASS_MAP = {0:0, 1:0, 2:1, 3:2, 4:2}
CLASS_NAMES = ['Low (0/1)', 'Medium (2)', 'High (3/4)']
KEYS = ['participant_id', 'submission_id']
N_OUTER, N_INNER = 5, 3
print(f'Python {platform.python_version()} | TensorFlow {tf.__version__} | seed={SEED}')
print('Task: three classes | primary validation: nested participant-grouped 5-fold CV')

Python 3.13.9 | TensorFlow 2.21.0 | seed=42
Task: three classes | primary validation: nested participant-grouped 5-fold CV


In [3]:
# 2. Locate data and output folders (Drive-mounted path in Colab; local path for this verified run)
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/MultiPENG/feature extracted datasets')
    OUTPUT_DIR = Path('/content/drive/MyDrive/MultiPENG/phase1_outputs')
except ImportError:
    IN_COLAB = False
    DATA_DIR = Path(r'/path/to/MultiPENG/feature extracted datasets')
    OUTPUT_DIR = Path.cwd() / 'output' / 'Phase1_Final_Run'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = OUTPUT_DIR / 'plots'; PLOT_DIR.mkdir(parents=True, exist_ok=True)
FILES = {'EEG':'EEG_prepared.csv','Eye':'EYE_prepared.csv','Face':'Face_prepared.csv','HR':'HR_prepared.csv'}
missing = [f for f in FILES.values() if not (DATA_DIR/f).exists()]
if missing:
    raise FileNotFoundError(f'Missing files under {DATA_DIR}: {missing}. In Colab, put the four CSVs in the Drive folder above; no files.upload() step is used.')
print('Data folder:', DATA_DIR)
print('Output folder:', OUTPUT_DIR)
print('Input files are read directly from the configured folder; no notebook upload widget is used.')

Data folder: [local path omitted]
Output folder: [local path omitted]
Input files are read directly from the configured folder; no notebook upload widget is used.


In [4]:
# 3. Load each modality and display its dimensions
raw = {m: pd.read_csv(DATA_DIR/f, low_memory=False) for m,f in FILES.items()}
inventory = pd.DataFrame([{'Modality':m, 'File':FILES[m], 'Rows':len(df), 'Columns':df.shape[1],
    'Participants':df['participant_id'].nunique() if 'participant_id' in df else np.nan,
    'Unique sessions':df['submission_id'].nunique() if 'submission_id' in df else np.nan,
    'Missing cells':int(df.isna().sum().sum())} for m,df in raw.items()])
display(inventory)
for m,df in raw.items(): print(f'{m}: {df.shape[0]:,} rows × {df.shape[1]:,} columns')

Modality              File  Rows  Columns  Participants  Unique sessions  Missing cells
     EEG  EEG_prepared.csv   873      818            38              873             55
     Eye  EYE_prepared.csv   873      163            38              873              0
    Face Face_prepared.csv   873     1673            38              873              0
      HR   HR_prepared.csv   873       18            38              873              9
EEG: 873 rows × 818 columns
Eye: 873 rows × 163 columns
Face: 873 rows × 1,673 columns
HR: 873 rows × 18 columns


In [5]:
# 4. Verify keys, labels, row uniqueness, and exact session synchronization
for m,df in raw.items():
    need = KEYS + ['engagement']
    absent = [c for c in need if c not in df.columns]
    if absent: raise ValueError(f'{m} missing required columns: {absent}')
    if df.duplicated(KEYS).any(): raise ValueError(f'{m} contains duplicate participant/session keys')
base = raw['EEG'][KEYS+['engagement']].copy()
sync_rows=[]
for m,df in raw.items():
    chk=base.merge(df[KEYS+['engagement']],on=KEYS,how='outer',suffixes=('_EEG','_mod'),indicator=True,validate='one_to_one')
    aligned=bool(chk['_merge'].eq('both').all())
    labels=bool(chk['engagement_EEG'].eq(chk['engagement_mod']).all()) if aligned else False
    sync_rows.append({'Modality':m,'Keys aligned':aligned,'Labels match':labels,'Duplicate key rows':int(df.duplicated(KEYS).sum()),'Exact duplicate rows':int(df.duplicated().sum())})
    if not aligned or not labels: raise ValueError(f'{m} does not align with EEG sessions and labels')
display(pd.DataFrame(sync_rows))
print(f'Validated {len(base):,} paired sessions and {base.participant_id.nunique()} participants.')

Modality  Keys aligned  Labels match  Duplicate key rows  Exact duplicate rows
     EEG          True          True                   0                     0
     Eye          True          True                   0                     0
    Face          True          True                   0                     0
      HR          True          True                   0                     0
Validated 873 paired sessions and 38 participants.


In [6]:
# 5. Feature/data-quality audit and target-leakage screening
quality=[]; feature_cols={}; merged=base.copy()
for m,df in raw.items():
    candidates=[c for c in df.columns if c not in KEYS+['engagement'] and
        not re.search(r'timestamp|counter|frame.?id|submission.?id|participant.?id',c,re.I) and
        pd.api.types.is_numeric_dtype(df[c])]
    # Prevent use of vendor-provided engagement scores as predictors.
    proxy=[c for c in candidates if 'engagement' in c.lower()]
    candidates=[c for c in candidates if c not in proxy]
    feature_cols[m]=candidates
    x=df[candidates].replace([np.inf,-np.inf],np.nan)
    const=[c for c in candidates if x[c].nunique(dropna=True)<=1]
    dup=int(x.T.duplicated().sum())
    numeric=x.select_dtypes(include=np.number).to_numpy(dtype='float64',copy=False)
    quality.append({'Modality':m,'Input numeric features':len(candidates)+len(proxy),'Engagement-proxy features excluded':len(proxy),
        'Missing feature cells':int(x.isna().sum().sum()),'Infinite values':int(np.isinf(numeric).sum()),
        'Constant/all-empty features':len(const),'Duplicated feature columns':dup,
        'Median abs skew (sample)':float(x.skew(numeric_only=True).abs().median()) if len(candidates) else np.nan})
    merged=merged.merge(df[KEYS+candidates].rename(columns={c:f'{m}__{c}' for c in candidates}),on=KEYS,validate='one_to_one')
quality_df=pd.DataFrame(quality); display(quality_df)
print('Participant/session IDs are retained for synchronization/grouping only, never as model predictors.')
print('EEG engagement-derived predictors are excluded to avoid direct target leakage.')

Modality  Input numeric features  Engagement-proxy features excluded  Missing feature cells  Infinite values  Constant/all-empty features  Duplicated feature columns  Median abs skew (sample)
     EEG                     800                                  25                      0                0                           71                          87                  4.718707
     Eye                     155                                   0                      0                0                           24                          34                  1.245562
    Face                    1665                                   0                      0                0                           42                          40                  0.606585
      HR                      10                                   0                      6                0                            0                           0                  1.058537
Participant/session IDs are retained for

In [7]:
# 6. Map target labels and show the original class imbalance
y_original=pd.to_numeric(merged.engagement,errors='raise').astype(int).to_numpy()
if not set(np.unique(y_original)).issubset(CLASS_MAP): raise ValueError('Expected original engagement labels 0..4')
y=np.array([CLASS_MAP[v] for v in y_original],dtype=int)
groups=merged.participant_id.astype(str).to_numpy()
class_counts=np.bincount(y,minlength=3)
class_table=pd.DataFrame({'Class':CLASS_NAMES,'Sessions':class_counts,'Percent':100*class_counts/len(y)})
display(class_table.style.format({'Percent':'{:.1f}%'}))
print(f'Class imbalance ratio (largest/smallest): {class_counts.max()/class_counts.min():.2f}:1')
fig,ax=plt.subplots(figsize=(7,4)); sns.barplot(data=class_table,x='Class',y='Sessions',color='#4C78A8',ax=ax)
ax.set_title('Original engagement distribution (before any training-only balancing)'); ax.set_ylabel('Sessions')
for i,row in class_table.iterrows(): ax.text(i,row.Sessions+5,f'{row.Sessions} ({row.Percent:.1f}%)',ha='center')
plt.tight_layout(); fig.savefig(PLOT_DIR/'01_class_distribution.png',dpi=180,bbox_inches='tight'); plt.show()

     Class  Sessions   Percent
 Low (0/1)        97 11.111111
Medium (2)       156 17.869416
High (3/4)       620 71.019473
Class imbalance ratio (largest/smallest): 6.39:1

[stderr]
notebook_cell_7:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


In [8]:
# 7. Define leakage-safe, fold-local preprocessing by modality
def prepare_modalities(train_idx, eval_idx):
    train_blocks={}; eval_blocks={}; kept_names={}
    for m in MODALITIES:
        cols=[f'{m}__{c}' for c in feature_cols[m]]
        A=merged.iloc[train_idx][cols].replace([np.inf,-np.inf],np.nan).astype('float32')
        B=merged.iloc[eval_idx][cols].replace([np.inf,-np.inf],np.nan).astype('float32')
        usable=[c for c in cols if A[c].notna().any() and A[c].nunique(dropna=True)>1]
        A=A[usable]; B=B[usable]
        dup=A.T.duplicated(); A=A.loc[:,~dup]; B=B.loc[:,~dup]
        if A.shape[1]==0: raise ValueError(f'No usable features remain for {m}')
        imp=SimpleImputer(strategy='median'); A=imp.fit_transform(A); B=imp.transform(B)
        scaler=StandardScaler(); A=scaler.fit_transform(A); B=scaler.transform(B)
        selector=SelectPercentile(score_func=f_classif,percentile=50)
        A=selector.fit_transform(A,y[train_idx]); B=selector.transform(B)
        train_blocks[m]=A.astype('float32'); eval_blocks[m]=B.astype('float32')
        kept_names[m]=int(A.shape[1])
    return train_blocks,eval_blocks,kept_names
print('Within each training split: invalid/constant/duplicate filtering → median imputation → standard scaling → top 50% ANOVA selection.')
print('Every fitted transform is applied to its validation fold without refitting or using held-out labels.')

Within each training split: invalid/constant/duplicate filtering → median imputation → standard scaling → top 50% ANOVA selection.
Every fitted transform is applied to its validation fold without refitting or using held-out labels.


In [9]:
# 8. Define identical outer and inner participant-grouped folds
outer_cv=StratifiedGroupKFold(n_splits=N_OUTER,shuffle=True,random_state=SEED)
outer_splits=list(outer_cv.split(np.zeros(len(y)),y,groups))
fold_plan=[]
for k,(tr,te) in enumerate(outer_splits,1):
    overlap=set(groups[tr]) & set(groups[te]); assert not overlap
    fold_plan.append({'Fold':k,'Train sessions':len(tr),'Held-out sessions':len(te),'Train participants':len(set(groups[tr])),
        'Held-out participants':len(set(groups[te])),'Held-out Low':int((y[te]==0).sum()),
        'Held-out Medium':int((y[te]==1).sum()),'Held-out High':int((y[te]==2).sum())})
display(pd.DataFrame(fold_plan))
print('Group integrity check passed: no participant crosses a train/held-out boundary.')

 Fold  Train sessions  Held-out sessions  Train participants  Held-out participants  Held-out Low  Held-out Medium  Held-out High
    1             685                188                  29                      9            24               33            131
    2             712                161                  31                      7            11               33            117
    3             694                179                  31                      7            28               30            121
    4             713                160                  31                      7            14               27            119
    5             688                185                  30                      8            20               33            132
Group integrity check passed: no participant crosses a train/held-out boundary.


In [10]:
# 9. Define compact gated four-encoder MLP and macro-F1 early stopping
class TakeGate(layers.Layer):
    def __init__(self,index,**kwargs): super().__init__(**kwargs); self.index=index
    def call(self,x): return x[:,self.index:self.index+1]
    def get_config(self): return {**super().get_config(),'index':self.index}

class MacroF1Stop(keras.callbacks.Callback):
    def __init__(self,xv,yv,patience=8): super().__init__(); self.xv=xv; self.yv=yv; self.patience=patience
        
    def on_train_begin(self,logs=None): self.best=-1; self.wait=0; self.best_epoch=1; self.best_weights=None
    def on_epoch_end(self,epoch,logs=None):
        pred=self.model.predict(self.xv,verbose=0).argmax(axis=1)
        score=f1_score(self.yv,pred,average='macro',zero_division=0)
        if score>self.best+1e-5:
            self.best=score; self.best_epoch=epoch+1; self.wait=0; self.best_weights=self.model.get_weights()
        else:
            self.wait+=1
            if self.wait>=self.patience: self.model.stop_training=True
    def on_train_end(self,logs=None):
        if self.best_weights is not None: self.model.set_weights(self.best_weights)

def make_gated_mlp(dims,dropout=.30):
    ins=[]; zs=[]
    for m in MODALITIES:
        x=keras.Input(shape=(dims[m],),name=f'{m}_input'); ins.append(x)
        z=layers.Dense(64,activation='gelu')(x); z=layers.LayerNormalization()(z); z=layers.Dropout(dropout)(z)
        z=layers.Dense(32,activation='gelu',name=f'{m}_embedding')(z); zs.append(z)
    joined=layers.Concatenate()(zs)
    logits=layers.Dense(32,activation='gelu')(joined)
    gate=layers.Softmax(name='modality_gate')(layers.Dense(len(MODALITIES))(logits))
    weighted=[layers.Multiply()([z,TakeGate(i)(gate)]) for i,z in enumerate(zs)]
    fused=layers.Concatenate(name='gated_fusion')(weighted)
    h=layers.Dense(64,activation='gelu')(fused); h=layers.Dropout(dropout)(h)
    out=layers.Dense(3,activation='softmax',name='engagement_probabilities')(h)
    return keras.Model(ins,out)

def train_gated(xtr,ytr,xv,yv,weight_mode,seed,max_epochs=45):
    keras.utils.set_random_seed(seed); model=make_gated_mlp({m:xtr[m].shape[1] for m in MODALITIES})
    counts=np.bincount(ytr,minlength=3).astype(float)
    raw=len(ytr)/(3*np.maximum(counts,1)); weights=(np.sqrt(raw) if weight_mode=='sqrt_weight' else raw); weights=weights/weights.mean()
    model.compile(optimizer=keras.optimizers.AdamW(learning_rate=8e-4,weight_decay=1e-3),loss='sparse_categorical_crossentropy')
    cb=MacroF1Stop([xv[m] for m in MODALITIES],yv,patience=8)
    model.fit([xtr[m] for m in MODALITIES],ytr,epochs=max_epochs,batch_size=64,verbose=0,
        class_weight={i:float(weights[i]) for i in range(3)},callbacks=[cb])
    return model,cb.best_epoch,cb.best
print('MLP: four Dense(64)→LayerNorm→Dropout→Dense(32) encoders; learned 4-way Softmax gate; Dense(64) fusion head; 3-class Softmax.')
print('Optimization: AdamW, class-weighted loss, fold-local Macro-F1 early stopping.')

MLP: four Dense(64)→LayerNorm→Dropout→Dense(32) encoders; learned 4-way Softmax gate; Dense(64) fusion head; 3-class Softmax.
Optimization: AdamW, class-weighted loss, fold-local Macro-F1 early stopping.


In [11]:
# 10. Candidate tabular models and training-only imbalance methods
SVM_CANDIDATES=[('SVM weighted C=1',{'balance':'class_weight','C':1.0}),
    ('SVM oversampled C=1',{'balance':'random_oversampling','C':1.0}),
    ('SVM unweighted C=2',{'balance':'none','C':2.0})]
RF_CANDIDATES=[('RF unweighted',{'balance':'none'}),('RF balanced_subsample',{'balance':'class_weight'})]
def fit_tabular(name,params,xtr,ytr,seed):
    X=np.concatenate([xtr[m] for m in MODALITIES],axis=1); yy=ytr.copy(); bal=params['balance']
    if bal=='random_oversampling': X,yy=random_oversample(X,yy,seed)
    if name.startswith('SVM'):
        model=SVC(C=params['C'],kernel='rbf',gamma='scale',class_weight='balanced' if bal=='class_weight' else None,probability=True,random_state=seed)
    else:
        model=RandomForestClassifier(n_estimators=250,min_samples_leaf=2,max_features='sqrt',
            class_weight='balanced_subsample' if bal=='class_weight' else None,n_jobs=-1,random_state=seed)
    model.fit(X,yy); return model
def tabular_prob(model,blocks): return model.predict_proba(np.concatenate([blocks[m] for m in MODALITIES],axis=1))
RUN_GATED_MLP = bool(tf.config.list_physical_devices('GPU'))
def random_oversample(X,y,seed):
    rng=np.random.default_rng(seed); counts=np.bincount(y,minlength=3); target=int(counts.max()); take=[]
    for cls in range(3):
        ids=np.flatnonzero(y==cls)
        if len(ids)==0: raise ValueError(f'Cannot oversample absent class {cls}')
        take.extend(ids.tolist())
        if len(ids)<target: take.extend(rng.choice(ids,size=target-len(ids),replace=True).tolist())
    take=np.asarray(take); rng.shuffle(take); return X[take],y[take]
print('SVM and RF balance options are selected on inner participant folds only; training-only random oversampling is implemented locally and validation rows remain untouched.')
print('Gated MLP execution enabled:', RUN_GATED_MLP, '| local CPU runs skip repeated neural fitting; Colab GPU runs include it.')

SVM and RF balance options are selected on inner participant folds only; training-only random oversampling is implemented locally and validation rows remain untouched.
Gated MLP execution enabled: False | local CPU runs skip repeated neural fitting; Colab GPU runs include it.


In [12]:
# 11. Nested model selection and honest outer out-of-fold predictions
t0=time.time(); oof_pred=np.full(len(y),-1,dtype=int); oof_prob=np.full((len(y),3),np.nan,dtype='float32')
oof_fold=np.zeros(len(y),dtype=int); oof_model=np.full(len(y),'',dtype=object)
outer_rows=[]; candidate_rows=[]; feature_rows=[]; fold_gate_rows=[]
for ofold,(otr,ote) in enumerate(outer_splits,1):
    print(f'\nOUTER FOLD {ofold}/{N_OUTER}: fitting model selection on {len(otr)} sessions; scoring {len(ote)} held-out sessions')
    inner_cv=StratifiedGroupKFold(n_splits=N_INNER,shuffle=True,random_state=SEED+ofold)
    inner_splits=list(inner_cv.split(np.zeros(len(otr)),y[otr],groups[otr]))
    inner_cache=[]
    for j,(itr_rel,iva_rel) in enumerate(inner_splits,1):
        itr,iva=otr[itr_rel],otr[iva_rel]
        xtr,xv,kept=prepare_modalities(itr,iva); inner_cache.append((xtr,y[itr],xv,y[iva],SEED+ofold*100+j))
    scores=[]
    for name,params in SVM_CANDIDATES+RF_CANDIDATES:
        vals=[]
        for xtr,ytr,xv,yv,seed in inner_cache:
            model=fit_tabular(name,params,xtr,ytr,seed); prob=tabular_prob(model,xv); pred=prob.argmax(axis=1)
            vals.append(f1_score(yv,pred,average='macro',zero_division=0)); del model
        rec={'Outer fold':ofold,'Candidate':name,'Balance':params['balance'],'Mean inner Macro-F1':float(np.mean(vals)),'Inner fold SD':float(np.std(vals,ddof=1))}
        candidate_rows.append(rec); scores.append((rec['Mean inner Macro-F1'],name,params,None))
        print(f'  {name:25s} inner Macro-F1={rec["Mean inner Macro-F1"]:.3f} ± {rec["Inner fold SD"]:.3f}')
    if RUN_GATED_MLP:
        for mode in ['sqrt_weight','balanced_weight']:
            vals=[]; epochs=[]
            for j,(xtr,ytr,xv,yv,seed) in enumerate(inner_cache,1):
                model,ep,score=train_gated(xtr,ytr,xv,yv,mode,seed+900)
                vals.append(score); epochs.append(ep); del model; keras.backend.clear_session(); gc.collect()
            name=f'Gated MLP {mode}'; params={'weight_mode':mode,'epochs':int(np.median(epochs))}
            rec={'Outer fold':ofold,'Candidate':name,'Balance':mode,'Mean inner Macro-F1':float(np.mean(vals)),'Inner fold SD':float(np.std(vals,ddof=1)),'Selected epochs':params['epochs']}
            candidate_rows.append(rec); scores.append((rec['Mean inner Macro-F1'],name,params,epochs))
            print(f'  {name:25s} inner Macro-F1={rec["Mean inner Macro-F1"]:.3f} ± {rec["Inner fold SD"]:.3f}; median best epoch={params["epochs"]}')
    else:
        print('  Gated MLP skipped for this CPU-only execution; implementation remains available for a Colab GPU run.')
    best=max(scores,key=lambda z:(z[0],-len(z[1])))
    best_inner,name,params,_=best
    xtr,xte,kept=prepare_modalities(otr,ote); feature_rows.append({'Outer fold':ofold,**kept})
    if name.startswith('Gated MLP'):
        model=make_gated_mlp({m:xtr[m].shape[1] for m in MODALITIES})
        n=np.bincount(y[otr],minlength=3).astype(float); w=len(otr)/(3*np.maximum(n,1))
        w=np.sqrt(w) if params['weight_mode']=='sqrt_weight' else w; w=w/w.mean()
        model.compile(optimizer=keras.optimizers.AdamW(learning_rate=8e-4,weight_decay=1e-3),loss='sparse_categorical_crossentropy')
        model.fit([xtr[m] for m in MODALITIES],y[otr],epochs=max(1,params['epochs']),batch_size=64,verbose=0,
            class_weight={i:float(w[i]) for i in range(3)})
        prob=model.predict([xte[m] for m in MODALITIES],verbose=0)
        gates=keras.Model(model.inputs,model.get_layer('modality_gate').output).predict([xte[m] for m in MODALITIES],verbose=0)
        fold_gate_rows.append({'Fold':ofold,**{m:float(gates[:,i].mean()) for i,m in enumerate(MODALITIES)}})
        del model; keras.backend.clear_session()
    else:
        model=fit_tabular(name,params,xtr,y[otr],SEED+ofold*10000); prob=tabular_prob(model,xte); del model
    pred=prob.argmax(axis=1); oof_pred[ote]=pred; oof_prob[ote]=prob; oof_fold[ote]=ofold; oof_model[ote]=name
    row={'Fold':ofold,'Selected model':name,'Selected balance':params['balance'],'Best inner Macro-F1':best_inner,
        'Accuracy':accuracy_score(y[ote],pred),'Macro-F1':f1_score(y[ote],pred,average='macro',zero_division=0),
        'Weighted-F1':f1_score(y[ote],pred,average='weighted',zero_division=0),
        'Balanced accuracy':balanced_accuracy_score(y[ote],pred),'MCC':matthews_corrcoef(y[ote],pred)}
    outer_rows.append(row)
    print(f'  SELECTED {name} | held-out accuracy={row["Accuracy"]:.3f} | Macro-F1={row["Macro-F1"]:.3f} | Weighted-F1={row["Weighted-F1"]:.3f}')
    del xtr,xte; gc.collect()
assert np.all(oof_fold>0) and np.isfinite(oof_prob).all()
print(f'\nNested participant-grouped run complete in {(time.time()-t0)/60:.1f} minutes.')


OUTER FOLD 1/5: fitting model selection on 685 sessions; scoring 188 held-out sessions
  SVM weighted C=1          inner Macro-F1=0.279 ± 0.006
  SVM oversampled C=1       inner Macro-F1=0.321 ± 0.024
  SVM unweighted C=2        inner Macro-F1=0.302 ± 0.027
  RF unweighted             inner Macro-F1=0.285 ± 0.010
  RF balanced_subsample     inner Macro-F1=0.282 ± 0.010
  Gated MLP skipped for this CPU-only execution; implementation remains available for a Colab GPU run.
  SELECTED SVM oversampled C=1 | held-out accuracy=0.707 | Macro-F1=0.440 | Weighted-F1=0.655

OUTER FOLD 2/5: fitting model selection on 712 sessions; scoring 161 held-out sessions
  SVM weighted C=1          inner Macro-F1=0.275 ± 0.001
  SVM oversampled C=1       inner Macro-F1=0.344 ± 0.026
  SVM unweighted C=2        inner Macro-F1=0.306 ± 0.032
  RF unweighted             inner Macro-F1=0.293 ± 0.023
  RF balanced_subsample     inner Macro-F1=0.280 ± 0.010
  Gated MLP skipped for this CPU-only execution; implemen

In [13]:
# 12. Print selected model, fold results, and aggregate out-of-fold metrics
fold_metrics=pd.DataFrame(outer_rows); candidate_metrics=pd.DataFrame(candidate_rows); feature_counts=pd.DataFrame(feature_rows)
def metric_summary(truth,pred):
    return {'Sessions':len(truth),'Accuracy':accuracy_score(truth,pred),'Macro-F1':f1_score(truth,pred,average='macro',zero_division=0),
        'Weighted-F1':f1_score(truth,pred,average='weighted',zero_division=0),'Micro-F1':f1_score(truth,pred,average='micro',zero_division=0),
        'Balanced accuracy':balanced_accuracy_score(truth,pred),'MCC':matthews_corrcoef(truth,pred)}
overall=pd.DataFrame([{'System':'Nested grouped model-selection procedure',**metric_summary(y,oof_pred)},
    {'System':'Majority-only reference',**metric_summary(y,np.full(len(y),int(np.bincount(y).argmax()))) }])
print('Outer-fold selected models and scores:'); display(fold_metrics.round(3))
print('Overall honest out-of-fold comparison:'); display(overall.round(3))
print('Mean ± SD across outer folds:'); display(fold_metrics[['Accuracy','Macro-F1','Weighted-F1','Balanced accuracy','MCC']].agg(['mean','std']).round(3))
print('Candidate inner-CV selection results:'); display(candidate_metrics.groupby('Candidate')['Mean inner Macro-F1'].agg(['mean','std']).sort_values('mean',ascending=False).round(3))

Outer-fold selected models and scores:
 Fold      Selected model    Selected balance  Best inner Macro-F1  Accuracy  Macro-F1  Weighted-F1  Balanced accuracy    MCC
    1 SVM oversampled C=1 random_oversampling                0.321     0.707     0.440        0.655              0.441  0.261
    2 SVM oversampled C=1 random_oversampling                0.344     0.677     0.346        0.619              0.373  0.085
    3 SVM oversampled C=1 random_oversampling                0.336     0.587     0.417        0.581              0.418  0.144
    4 SVM oversampled C=1 random_oversampling                0.395     0.619     0.257        0.573              0.277 -0.128
    5 SVM oversampled C=1 random_oversampling                0.343     0.692     0.293        0.603              0.331  0.061
Overall honest out-of-fold comparison:
                                  System  Sessions  Accuracy  Macro-F1  Weighted-F1  Micro-F1  Balanced accuracy   MCC
Nested grouped model-selection procedure       

In [14]:
# 13. Print full per-class metrics and confusion matrix
report=classification_report(y,oof_pred,labels=[0,1,2],target_names=CLASS_NAMES,output_dict=True,zero_division=0)
per_class=pd.DataFrame(report).T.reset_index().rename(columns={'index':'Class'})
print('Per-class precision, recall, F1, and support from outer out-of-fold predictions:')
display(per_class[per_class['Class'].isin(CLASS_NAMES)].round(3))
cm=confusion_matrix(y,oof_pred,labels=[0,1,2]); cm_df=pd.DataFrame(cm,index=CLASS_NAMES,columns=CLASS_NAMES)
print('Confusion matrix (rows=actual, columns=predicted):'); display(cm_df)
fig,ax=plt.subplots(figsize=(7,5)); sns.heatmap(cm_df,annot=True,fmt='d',cmap='Blues',cbar=False,ax=ax)
ax.set_title('Nested participant-grouped out-of-fold confusion matrix'); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); fig.savefig(PLOT_DIR/'02_oof_confusion_matrix.png',dpi=180,bbox_inches='tight'); plt.show()

Per-class precision, recall, F1, and support from outer out-of-fold predictions:
     Class  precision  recall  f1-score  support
 Low (0/1)      0.265   0.134     0.178     97.0
Medium (2)      0.184   0.103     0.132    156.0
High (3/4)      0.739   0.879     0.803    620.0
Confusion matrix (rows=actual, columns=predicted):
 Low (0/1)  Medium (2)  High (3/4)
        13          20          64
        12          16         128
        24          51         545

[stderr]
notebook_cell_14:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


In [15]:
# 14. Plot the full metrics and fold stability for DSS/review use
plot_metrics=['Accuracy','Macro-F1','Weighted-F1','Balanced accuracy']
fig,ax=plt.subplots(1,2,figsize=(13,4.8))
overall.set_index('System')[plot_metrics].plot(kind='bar',ax=ax[0],ylim=(0,1),color=['#4C78A8','#F58518','#54A24B','#B279A2'])
ax[0].set_title('Out-of-fold model vs majority reference'); ax[0].set_ylabel('Score'); ax[0].tick_params(axis='x',rotation=12)
fold_metrics.set_index('Fold')[plot_metrics].plot(marker='o',ax=ax[1]); ax[1].set_ylim(0,1); ax[1].set_title('Participant-grouped outer-fold variability'); ax[1].set_ylabel('Score'); ax[1].set_xticks(fold_metrics['Fold'])
plt.tight_layout(); fig.savefig(PLOT_DIR/'03_metrics_and_fold_stability.png',dpi=180,bbox_inches='tight'); plt.show()
fig,ax=plt.subplots(figsize=(8,4.5)); per=per_class[per_class['Class'].isin(CLASS_NAMES)].set_index('Class')
per[['precision','recall','f1-score']].plot(kind='bar',ax=ax,color=['#4C78A8','#F58518','#54A24B'])
ax.set_ylim(0,1); ax.set_title('Per-class results: minority-class performance matters'); ax.set_ylabel('Score'); ax.tick_params(axis='x',rotation=0)
plt.tight_layout(); fig.savefig(PLOT_DIR/'04_per_class_metrics.png',dpi=180,bbox_inches='tight'); plt.show()


[stderr]
notebook_cell_15:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
notebook_cell_15:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


In [16]:
# 15. Inspect learned dynamic gate weights when gated MLP was selected
gate_df=pd.DataFrame(fold_gate_rows)
if len(gate_df):
    display(gate_df.round(3))
    means=gate_df[MODALITIES].mean().sort_values(ascending=False)
    fig,ax=plt.subplots(figsize=(7,4)); means.plot(kind='bar',color='#72B7B2',ax=ax)
    ax.set_ylim(0,1); ax.set_title('Mean gated-MLP modality weights on held-out sessions'); ax.set_ylabel('Mean gate weight'); plt.xticks(rotation=0)
    plt.tight_layout(); fig.savefig(PLOT_DIR/'05_mean_modality_gate_weights.png',dpi=180,bbox_inches='tight'); plt.show()
else:
    print('Gated MLP was not selected in any outer fold; no gate plot is reported. This is a valid model-selection outcome.')

Gated MLP was not selected in any outer fold; no gate plot is reported. This is a valid model-selection outcome.


In [17]:
# 16. Export exact Phase 1 prediction outputs for Phase 2
predictions=pd.DataFrame({'participant_id':groups,'session_id':merged.submission_id.astype(str),
    'actual_engagement_grouped_0_2':y,'actual_engagement_original_0_4':y_original,
    'predicted_engagement':oof_pred,'predicted_engagement_name':[CLASS_NAMES[i] for i in oof_pred],
    'validation_fold':oof_fold,'prediction_method':oof_model,'prediction_protocol':'nested_participant_grouped_5fold_oof',
    'confidence':oof_prob.max(axis=1),'probability_low':oof_prob[:,0],'probability_medium':oof_prob[:,1],'probability_high':oof_prob[:,2]})
pred_path=OUTPUT_DIR/'phase1_participant_grouped_oof_predictions.csv'
predictions.to_csv(pred_path,index=False)
print(f'Saved {len(predictions):,} out-of-fold session predictions to: {pred_path}')
display(predictions.head(10))

Out-of-fold predictions saved for 873 sessions.
Per-session participant/session identifiers and prediction rows are omitted from this public preview.


In [18]:
# 17. Save metrics, tables, plots, configuration, checksums, and a Phase 2 bundle
fold_metrics.to_csv(OUTPUT_DIR/'outer_fold_metrics.csv',index=False)
candidate_metrics.to_csv(OUTPUT_DIR/'inner_candidate_scores.csv',index=False)
overall.to_csv(OUTPUT_DIR/'overall_metrics.csv',index=False)
per_class.to_csv(OUTPUT_DIR/'per_class_metrics.csv',index=False)
cm_df.to_csv(OUTPUT_DIR/'confusion_matrix.csv')
class_table.to_csv(OUTPUT_DIR/'class_distribution.csv',index=False)
quality_df.to_csv(OUTPUT_DIR/'data_quality_audit.csv',index=False)
feature_counts.to_csv(OUTPUT_DIR/'selected_feature_counts_by_fold.csv',index=False)
if len(gate_df): gate_df.to_csv(OUTPUT_DIR/'modality_gate_weights_by_fold.csv',index=False)
config={'seed':SEED,'target_mapping':{'0,1':'Low','2':'Medium','3,4':'High'},'n_outer_folds':N_OUTER,'n_inner_folds':N_INNER,
    'group_column':'participant_id','session_key':'submission_id','feature_selection':'ANOVA top 50% per modality, fitted inside each training fold',
    'preprocessing':'training-fold-only invalid/constant/duplicate removal, median imputation, standard scaling',
    'imbalance':'inner-fold comparison of class weights and random oversampling; apply to training split only',
    'candidate_families_evaluated_on_this_run':['RBF-SVM','Random Forest'],
    'gated_four_encoder_mlp':'defined in notebook; executed only when a GPU is available',
    'metrics':'Accuracy, Macro-F1, Weighted-F1, Micro-F1, balanced accuracy, MCC, per-class scores, confusion matrix',
    'prediction_file':pred_path.name,'output_dir':str(OUTPUT_DIR),'data_dir':str(DATA_DIR),
    'data_rows':len(y),'participants':int(pd.Series(groups).nunique()),'framework':'Nested participant-grouped out-of-fold validation; not a locked test set'}
(OUTPUT_DIR/'experiment_config.json').write_text(json.dumps(config,indent=2),encoding='utf-8')
lines=['# Phase 1 run summary','',f'- Sessions: {len(y)}',f'- Participants: {pd.Series(groups).nunique()}','- Target: Low (0/1), Medium (2), High (3/4)','- Evaluation: nested 5-fold participant-grouped out-of-fold predictions','- Test-set claim: none; these are cross-validation estimates.','', '## Overall metrics',overall.to_markdown(index=False),'','## Outer folds',fold_metrics.to_markdown(index=False),'','## Per-class metrics',per_class[per_class['Class'].isin(CLASS_NAMES)].to_markdown(index=False),'','## Interpretation','Compare accuracy to the majority reference and use Macro-F1/per-class recall to judge imbalance. The Phase 2 CSV contains one outer-fold prediction per session; do not interpret it as a final untouched test set.']
(OUTPUT_DIR/'phase1_run_summary.md').write_text('\n'.join(lines),encoding='utf-8')
manifest=[]
for p in sorted(OUTPUT_DIR.rglob('*')):
    if p.is_file() and p.name!='phase1_dss_bundle.zip': manifest.append({'file':str(p.relative_to(OUTPUT_DIR)),'bytes':p.stat().st_size,'sha256':hashlib.sha256(p.read_bytes()).hexdigest()})
(OUTPUT_DIR/'artifact_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
zip_path=OUTPUT_DIR/'phase1_dss_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for name in ['phase1_participant_grouped_oof_predictions.csv','overall_metrics.csv','outer_fold_metrics.csv','inner_candidate_scores.csv','per_class_metrics.csv','confusion_matrix.csv','class_distribution.csv','data_quality_audit.csv','selected_feature_counts_by_fold.csv','experiment_config.json','phase1_run_summary.md','artifact_manifest.json','Phase1_Final_Four_Modality_Experiment.ipynb']:
        p=OUTPUT_DIR/name
        if p.exists(): z.write(p,arcname=name)
    for p in PLOT_DIR.glob('*.png'): z.write(p,arcname=f'plots/{p.name}')
print('Saved output tables, PNG plots, run configuration, summary, checksums, and Phase 2 bundle:')
for p in sorted(OUTPUT_DIR.iterdir()): print(' -',p.name, f'({p.stat().st_size:,} bytes)' if p.is_file() else '(folder)')


Saved output tables, PNG plots, run configuration, summary, checksums, and Phase 2 bundle:
 - artifact_manifest.json (2,691 bytes)
 - class_distribution.csv (125 bytes)
 - confusion_matrix.csv (97 bytes)
 - data_quality_audit.csv (344 bytes)
 - experiment_config.json (1,268 bytes)
 - inner_candidate_scores.csv (1,823 bytes)
 - outer_fold_metrics.csv (846 bytes)
 - overall_metrics.csv (366 bytes)
 - per_class_metrics.csv (501 bytes)
 - phase1_dss_bundle.zip (273,563 bytes)
 - Phase1_Final_Four_Modality_Experiment.ipynb (53,456 bytes)
 - phase1_participant_grouped_oof_predictions.csv (112,417 bytes)
 - phase1_run_summary.md (2,561 bytes)
 - plots (folder)
 - selected_feature_counts_by_fold.csv (108 bytes)


## How to read the saved result

`phase1_participant_grouped_oof_predictions.csv` is the Phase 2 prediction input: one session per row, including actual and predicted grouped labels, original label, participant/session keys, outer fold, selected model, confidence, and class probabilities. The `prediction_method` column states which candidate was selected in each outer fold.

The CSV contains **out-of-fold cross-validation predictions**, not an untouched test-set score and not serialized trained models. The ZIP includes this CSV, metric tables, plots, and configuration. When running this notebook in Colab, put the four source CSVs in the configured Google Drive folder; mount Drive and read them directly rather than using `files.upload()`.